# Bank Marketing — Customer Term-Deposit Subscription Classification

**BITS Pilani | M.Tech (DSE) | Machine Learning Assignment 2**

This notebook follows an end-to-end workflow: dataset inspection, preprocessing, exploratory analysis, model tuning, evaluation, fit diagnostics and artifact export.

**Prediction task:** predict whether a customer subscribes to a term deposit (`y = yes`) or does not (`y = no`).

> This is a binary classification problem. The positive class is `yes`. The feature `duration` is deliberately excluded because it records call duration and is not available before a call; using it would create post-event information leakage.


## 1. Setup and reproducibility

In [ ]:
from pathlib import Path
import io
import zipfile
import urllib.request
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

RANDOM_STATE = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "model"
REPORT_DIR = ROOT / "reports"
for folder in (DATA_DIR, MODEL_DIR, REPORT_DIR):
    folder.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)


## 2. Load the public UCI dataset

In [ ]:
# UCI Bank Marketing `bank-full` archive (45,211 rows; semicolon-separated CSV).
DATA_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
csv_path = DATA_DIR / "bank-full.csv"

if not csv_path.exists():
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=90) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        csv_member = next(name for name in archive.namelist() if name.endswith("bank-full.csv"))
        csv_path.write_bytes(archive.read(csv_member))

bank = pd.read_csv(csv_path, sep=";")
print("Dataset shape:", bank.shape)
display(bank.head())
display(bank.info())


## 3. Data understanding and quality checks

In [ ]:
print("Rows and columns:", bank.shape)
print("\\nMissing values per column:")
display(bank.isna().sum().to_frame("missing_values"))
print("\\nDuplicate rows:", bank.duplicated().sum())
print("\\nTarget counts:")
display(bank["y"].value_counts().rename_axis("subscription").to_frame("count"))
display((bank["y"].value_counts(normalize=True) * 100).round(2).rename("percentage").to_frame())


## 4. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=bank, x="y", ax=axes[0])
axes[0].set_title("Target class distribution")
axes[0].set_xlabel("Term-deposit subscription")
sns.histplot(data=bank, x="age", hue="y", bins=35, element="step",
             stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Age distribution by target")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=bank, x="y", y="balance", showfliers=False, ax=axes[0])
axes[0].set_title("Account balance by target")
sns.countplot(data=bank, x="month", hue="y",
              order=["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"],
              ax=axes[1])
axes[1].set_title("Campaign contacts by month")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 5. Feature selection and target definition

In [ ]:
# Exclude `duration`: it is only known after the call and is not suitable for a pre-call prediction.
TARGET = "y"
excluded_columns = [TARGET, "duration"]
X_all = bank.drop(columns=excluded_columns)
y_all = bank[TARGET].map({"no": 0, "yes": 1})

assert len(bank) >= 500, "The assignment requires at least 500 instances."
assert X_all.shape[1] >= 12, "The assignment requires at least 12 input features."
assert y_all.notna().all()

print("Instances:", len(bank))
print("Predictor features used:", X_all.shape[1])
print("Predictor columns:", X_all.columns.tolist())
print("Target labels: 0 = no, 1 = yes")


## 6. Sampling and train-test split

The full UCI file is relatively large for repeated cross-validation with kNN and SVM. To keep the experiment reproducible and practical, this notebook takes a **stratified random sample of 12,000 records** from the 45,211 available records. This is well above the assignment minimum of 500. The sample preserves the original target-class proportions.

An 80:20 stratified split is then used. The test set is kept separate until final evaluation.


In [ ]:
# Stratified sample from the complete dataset; fixed seed makes the selection reproducible.
sampled_bank = (
    bank.groupby("y", group_keys=False)
        .apply(lambda part: part.sample(
            n=round(12000 * len(part) / len(bank)),
            random_state=RANDOM_STATE
        ))
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
)
X = sampled_bank.drop(columns=excluded_columns)
y = sampled_bank[TARGET].map({"no": 0, "yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Sample size:", len(sampled_bank))
print("Training shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train target proportions:")
display(y_train.value_counts(normalize=True).rename(index={0:"no",1:"yes"}))
print("Test target proportions:")
display(y_test.value_counts(normalize=True).rename(index={0:"no",1:"yes"}))


## 7. Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 8. Model training and hyperparameter tuning

The five required algorithms are tuned using **5-fold StratifiedKFold cross-validation**. Macro-F1 is used as the tuning score so that performance on the minority subscription class is not hidden by the majority class.

All models use the same training and test split. Preprocessing is included inside each pipeline, so imputers, scaling and one-hot encoding are fitted on training folds only.


In [ ]:
models_and_grids = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear"),
        {"classifier__C": [0.1, 1.0, 10.0]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"classifier__max_depth": [4, 8, 12],
         "classifier__min_samples_leaf": [10, 30]}
    ),
    "K-Nearest Neighbors": (
        KNeighborsClassifier(),
        {"classifier__n_neighbors": [7, 15, 25],
         "classifier__weights": ["uniform", "distance"]}
    ),
    "Gaussian Naive Bayes": (
        GaussianNB(),
        {"classifier__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3]}
    ),
    "Support Vector Machine": (
        SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE),
        {"classifier__C": [0.5, 2.0, 10.0],
         "classifier__gamma": ["scale", 0.01]}
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
best_models, cv_summary = {}, {}

for model_name, (estimator, param_grid) in models_and_grids.items():
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", estimator)
    ])
    search = GridSearchCV(
        pipe, param_grid, scoring="f1_macro", cv=cv,
        n_jobs=-1, refit=True, verbose=1
    )
    print("\\nTuning:", model_name)
    search.fit(X_train, y_train)
    best_models[model_name] = search.best_estimator_
    cv_summary[model_name] = {
        "Best parameters": search.best_params_,
        "CV macro-F1 mean": search.best_score_,
        "CV macro-F1 std": search.cv_results_["std_test_score"][search.best_index_]
    }
    print("Best parameters:", search.best_params_)
    print("Best CV macro-F1:", round(search.best_score_, 4))


## 9. Evaluation on the held-out test set

The assignment asks for Accuracy, AUC, Precision, Recall, F1 and MCC. Because this is a binary classification problem, Precision, Recall and F1 are calculated for the positive `yes` class. ROC-AUC uses the predicted probability for `yes`. MCC summarizes the confusion matrix and accounts for all four outcomes.


In [ ]:
def calculate_metrics(model, features, target):
    predicted = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]
    return {
        "Accuracy": accuracy_score(target, predicted),
        "AUC": roc_auc_score(target, probabilities),
        "Precision": precision_score(target, predicted, zero_division=0),
        "Recall": recall_score(target, predicted, zero_division=0),
        "F1": f1_score(target, predicted, zero_division=0),
        "MCC": matthews_corrcoef(target, predicted)
    }

metric_rows = []
for name, fitted_model in best_models.items():
    row = {"ML Model Name": name}
    row.update(calculate_metrics(fitted_model, X_test, y_test))
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows).set_index("ML Model Name")
metrics_df = metrics_df[["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]].round(4)
display(metrics_df)
metrics_df.to_csv(REPORT_DIR / "model_comparison.csv")


## 10. Model comparison visualization

In [ ]:
metrics_df.plot(kind="bar", figsize=(12, 5), ylim=(0, 1), colormap="viridis")
plt.title("Comparison of five classification models on the test set")
plt.ylabel("Metric score")
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right", ncol=3)
plt.tight_layout()
plt.show()


## 11. Over-fitting and under-fitting analysis

In [ ]:
fit_rows = []
for name, fitted_model in best_models.items():
    train_pred = fitted_model.predict(X_train)
    test_pred = fitted_model.predict(X_test)
    train_f1 = f1_score(y_train, train_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)
    fit_rows.append({
        "Model": name,
        "Train F1": train_f1,
        "Test F1": test_f1,
        "Train-Test F1 gap": train_f1 - test_f1,
        "CV macro-F1 mean": cv_summary[name]["CV macro-F1 mean"],
        "CV macro-F1 std": cv_summary[name]["CV macro-F1 std"]
    })
fit_df = pd.DataFrame(fit_rows).set_index("Model").round(4)
display(fit_df)


## 12. Confusion matrix and classification report for the leading model

In [ ]:
winner_name = metrics_df["F1"].idxmax()
winner_model = best_models[winner_name]
winner_predictions = winner_model.predict(X_test)

cm = confusion_matrix(y_test, winner_predictions, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted no", "Predicted yes"],
            yticklabels=["Actual no", "Actual yes"])
plt.title(f"Confusion matrix — {winner_name}")
plt.tight_layout()
plt.show()

print("Leading model by positive-class F1:", winner_name)
print(classification_report(
    y_test, winner_predictions, labels=[0, 1],
    target_names=["no", "yes"], zero_division=0
))


## 13. Save models, test data and reports

In [ ]:
# Save complete fitted pipelines. Each includes its fitted preprocessing steps.
model_file_names = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "K-Nearest Neighbors": "knn.pkl",
    "Gaussian Naive Bayes": "naive_bayes.pkl",
    "Support Vector Machine": "support_vector_machine.pkl"
}
for name, fitted_model in best_models.items():
    joblib.dump(fitted_model, MODEL_DIR / model_file_names[name])

# The app evaluates these held-out raw rows; labels are retained to calculate metrics.
test_export = X_test.copy()
test_export["y"] = y_test.map({0: "no", 1: "yes"})
test_export.to_csv(ROOT / "test_data.csv", index=False)

pd.DataFrame(cv_summary).T.to_csv(REPORT_DIR / "cross_validation_summary.csv")
fit_df.to_csv(REPORT_DIR / "fit_diagnostics.csv")
winner_summary = (
    f"Leading model by positive-class F1: {winner_name}\n"
    f"F1: {metrics_df.loc[winner_name, 'F1']:.4f}\n"
    f"AUC: {metrics_df.loc[winner_name, 'AUC']:.4f}\n"
    f"MCC: {metrics_df.loc[winner_name, 'MCC']:.4f}\n"
)
(REPORT_DIR / "winner.txt").write_text(winner_summary, encoding="utf-8")

print("Saved five fitted models to:", MODEL_DIR)
print("Saved test data to:", ROOT / "test_data.csv")
print("Saved comparison and diagnostics to:", REPORT_DIR)


## 14. Interpretation guide for the assignment report

Use the actual tables above to write the observations. Do not assume that a particular algorithm will win before seeing the measured results.

- **Logistic Regression:** a linear baseline; compare recall and F1 for the `yes` class after class balancing.
- **Decision Tree:** can learn non-linear decision rules; compare train/test F1 gap to assess over-fitting.
- **KNN:** relies on distances, so numerical scaling is important; the one-hot feature space may affect distance quality.
- **Gaussian Naive Bayes:** trains quickly but assumes conditional independence between features; correlated customer/campaign variables can limit its fit.
- **Support Vector Machine:** the RBF kernel can learn non-linear boundaries; compare its AUC, F1 and MCC against training cost and fit gap.

**Overall winner:** choose the model with the highest positive-class F1 if identifying subscribers is the primary goal, and discuss its AUC, recall and MCC as supporting evidence. If your business objective prioritizes avoiding missed subscribers, explicitly discuss recall as well.


## 15. Final checklist

- [ ] All five required models trained on the same Bank Marketing sample.
- [ ] Six required metrics calculated and saved.
- [ ] Actual comparison table and observations added to README and PDF.
- [ ] Five fitted model files and `test_data.csv` saved.
- [ ] Streamlit app tested locally with the test CSV.
- [ ] Repository pushed to GitHub and link verified.
- [ ] App deployed on Streamlit Community Cloud and live link verified.
- [ ] Project executed in BITS Virtual Lab and one genuine screenshot captured.
- [ ] Final single PDF contains GitHub link, live app link, BITS screenshot and README content in the prescribed order.
